# F1 Strategy Dataset: Pit Stop Prediction (Complete Kaggle Workflow)

This notebook follows a full machine learning workflow to predict **`PitNextLap`** (whether a driver will pit on the next lap).

Workflow covered:
1. Import libraries
2. Load data
3. Basic exploration
4. Data cleaning
5. EDA
6. Feature engineering
7. Train/test split
8. Train at least 2 models
9. Evaluate models
10. Compare and select best model
11. Optional hyperparameter tuning
12. Final predictions and insights

## 1) Import Libraries

In [ ]:
# Core data libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Modeling and preprocessing
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

# Plot styling
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 200)

## 2) Load Dataset

In [ ]:
# Resolve dataset path for Kaggle and local environments
candidate_paths = [
    Path('/kaggle/input/formula1-dataset/f1_strategy_dataset_v1.csv'),
    Path('/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v1.csv'),
    Path('/kaggle/working/f1_strategy_dataset_v1.csv'),
    Path('./f1_strategy_dataset_v1.csv'),
    Path('/Users/berk.terekli/Documents/kaggle-notebooks/formula1-dataset/f1_strategy_dataset_v1.csv'),
]

data_path = None
for p in candidate_paths:
    if p.exists():
        data_path = p
        break

# Fallback search under /kaggle/input if needed
if data_path is None:
    matches = list(Path('/kaggle/input').glob('**/f1_strategy_dataset_v1.csv')) if Path('/kaggle/input').exists() else []
    if matches:
        data_path = matches[0]

if data_path is None:
    raise FileNotFoundError('Could not find f1_strategy_dataset_v1.csv. Please check dataset attachment/path.')

print(f'Using dataset: {data_path}')
df = pd.read_csv(data_path)
df.head()

## 3) Basic Data Exploration

In [ ]:
print('Shape:', df.shape)
print('\nColumns:\n', list(df.columns))
print('\nData types:\n')
print(df.dtypes)

print('\nMissing values by column:')
missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame('missing_count'))

print('\nSummary statistics (numeric):')
display(df.describe().T)

print('\nSummary statistics (categorical):')
display(df.describe(include='object').T)

## 4) Data Cleaning

In [ ]:
# Work on a copy to keep raw data untouched
df_clean = df.copy()

# Standardize column names (remove spaces/special characters)
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.replace(r'[^\w]+', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

# Remove exact duplicates
dup_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
dup_after = df_clean.duplicated().sum()

print(f'Duplicates before: {dup_before} | after: {dup_after}')

# Ensure expected target exists
target_col = 'PitNextLap'
if target_col not in df_clean.columns:
    raise ValueError(f"Target column '{target_col}' not found after cleaning. Columns: {df_clean.columns.tolist()}")

# Convert target to numeric and keep valid rows only
df_clean[target_col] = pd.to_numeric(df_clean[target_col], errors='coerce')
df_clean = df_clean[df_clean[target_col].notna()].copy()
df_clean[target_col] = df_clean[target_col].astype(int)

# Try converting numeric-like columns safely
for col in df_clean.columns:
    if col == target_col:
        continue
    # Convert only where numeric interpretation is meaningful
    if df_clean[col].dtype == 'object':
        converted = pd.to_numeric(df_clean[col], errors='coerce')
        # If at least 70% values are numeric after conversion, keep numeric type
        if converted.notna().mean() >= 0.70:
            df_clean[col] = converted

# Handle missing values
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()

for col in numeric_cols:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    if df_clean[col].isna().any():
        mode_val = df_clean[col].mode(dropna=True)
        fill_val = mode_val.iloc[0] if len(mode_val) else 'Unknown'
        df_clean[col] = df_clean[col].fillna(fill_val)

print('Missing values after cleaning:', int(df_clean.isna().sum().sum()))
display(df_clean.head())

## 5) Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='PitNextLap', data=df_clean, palette='Set2')
plt.title('Target Distribution: PitNextLap')
plt.xlabel('Pit Next Lap')
plt.ylabel('Count')
plt.show()

pit_rate = df_clean['PitNextLap'].mean()
print(f'PitNextLap positive rate: {pit_rate:.2%}')

In [ ]:
# Numeric feature distributions
numeric_cols = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c != 'PitNextLap']
plot_cols = numeric_cols[:8]  # keep plots readable

if plot_cols:
    n_cols = 2
    n_rows = int(np.ceil(len(plot_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    
    for i, col in enumerate(plot_cols):
        sns.histplot(df_clean[col], kde=True, ax=axes[i], color='steelblue')
        axes[i].set_title(f'Distribution: {col}')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap (numeric features)
num_df = df_clean.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 8))
cor_mat = num_df.corr(numeric_only=True)
sns.heatmap(cor_mat, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

# Most correlated features with target
if 'PitNextLap' in cor_mat.columns:
    target_corr = cor_mat['PitNextLap'].sort_values(ascending=False)
    print('Top correlations with PitNextLap:')
    display(target_corr.to_frame('correlation').head(10))

In [ ]:
# Relationship plots for key strategy variables (if available)
key_cols = ['TyreLife', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Compound']
available = [c for c in key_cols if c in df_clean.columns]

if 'Compound' in available:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=df_clean, x='Compound', hue='PitNextLap', palette='Set1')
    plt.title('PitNextLap by Tyre Compound')
    plt.xticks(rotation=30)
    plt.show()

for col in ['TyreLife', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress']:
    if col in available:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df_clean, x='PitNextLap', y=col, palette='Set3')
        plt.title(f'{col} vs PitNextLap')
        plt.show()

## 6) Feature Engineering

In [ ]:
# Create a feature-engineered copy
df_fe = df_clean.copy()

# Sort for sequential features
sort_cols = [c for c in ['Race', 'Driver', 'LapNumber'] if c in df_fe.columns]
if sort_cols:
    df_fe = df_fe.sort_values(sort_cols).reset_index(drop=True)

# Interaction feature: tire wear x degradation
if {'TyreLife', 'Cumulative_Degradation'}.issubset(df_fe.columns):
    df_fe['TyreLife_x_Degradation'] = df_fe['TyreLife'] * df_fe['Cumulative_Degradation']

# Late race indicator
if 'RaceProgress' in df_fe.columns:
    df_fe['IsLateRace'] = (df_fe['RaceProgress'] >= 0.75).astype(int)

# Previous lap time and rolling mean lap time per driver-race sequence
if {'Race', 'Driver', 'LapTime_s'}.issubset(df_fe.columns):
    df_fe['PrevLapTime_s'] = df_fe.groupby(['Race', 'Driver'])['LapTime_s'].shift(1)
    df_fe['RollingLapTime3_s'] = (
        df_fe.groupby(['Race', 'Driver'])['LapTime_s']
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

# Relative lap progress inside each driver-race stint
if {'Race', 'Driver', 'LapNumber'}.issubset(df_fe.columns):
    race_driver_max_lap = df_fe.groupby(['Race', 'Driver'])['LapNumber'].transform('max').replace(0, np.nan)
    df_fe['LapProgressWithinDriverRace'] = df_fe['LapNumber'] / race_driver_max_lap

# Fill any new missing numeric values
new_num_cols = df_fe.select_dtypes(include=[np.number]).columns
for col in new_num_cols:
    if df_fe[col].isna().any():
        df_fe[col] = df_fe[col].fillna(df_fe[col].median())

print('Feature engineering complete.')
print('Final dataset shape:', df_fe.shape)
print('New columns created:', [c for c in df_fe.columns if c not in df_clean.columns])

## 7) Train/Test Split

In [ ]:
target_col = 'PitNextLap'
X = df_fe.drop(columns=[target_col])
y = df_fe[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('Train target mean:', y_train.mean())
print('Test target mean :', y_test.mean())

## 8) Train At Least 2 Machine Learning Models

In [ ]:
# Define feature types
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Model definitions
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    )
}

# Fit model pipelines
fitted_models = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    print(f'Trained: {name}')

## 9) Evaluate Models

In [ ]:
def evaluate_model(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    # Probability scores for ROC-AUC when available
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_eval)[:, 1]
    else:
        # Fallback for models exposing decision_function
        raw_score = model.decision_function(X_eval)
        y_score = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min() + 1e-9)

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_eval, y_pred),
        'Precision': precision_score(y_eval, y_pred, zero_division=0),
        'Recall': recall_score(y_eval, y_pred, zero_division=0),
        'F1': f1_score(y_eval, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_eval, y_score),
    }

    return metrics, y_pred, y_score

results = []
pred_store = {}

for name, model in fitted_models.items():
    metrics, y_pred, y_score = evaluate_model(name, model, X_test, y_test)
    results.append(metrics)
    pred_store[name] = {'y_pred': y_pred, 'y_score': y_score}

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
# Confusion matrices and ROC curves
for name, model in fitted_models.items():
    y_pred = pred_store[name]['y_pred']

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    print(f'Classification Report - {name}')
    print(classification_report(y_test, y_pred, digits=4))

plt.figure(figsize=(7, 5))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=plt.gca())
plt.plot([0, 1], [0, 1], 'k--', alpha=0.6)
plt.title('ROC Curves')
plt.show()

## 10) Compare Results and Select Best Model

In [ ]:
# Select best model by F1 (useful under class imbalance)
best_baseline_row = results_df.iloc[0]
best_baseline_name = best_baseline_row['Model']
best_baseline_model = fitted_models[best_baseline_name]

print('Best baseline model (by F1):', best_baseline_name)
display(best_baseline_row.to_frame('value'))

## 11) Optional Hyperparameter Tuning (Random Forest)

In [ ]:
# Tune RandomForest if available in fitted models
rf_base = RandomForestClassifier(
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_base)
])

param_dist = {
    'model__n_estimators': [200, 300, 500, 700],
    'model__max_depth': [None, 6, 10, 14, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', None],
}

search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist,
    n_iter=15,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print('Best CV F1:', search.best_score_)

tuned_rf_model = search.best_estimator_
tuned_metrics, tuned_pred, tuned_score = evaluate_model('RandomForest_Tuned', tuned_rf_model, X_test, y_test)
print('\nTuned RF test metrics:')
print(tuned_metrics)

## 12) Final Predictions and Insights

In [ ]:
# Combine baseline and tuned result for final selection
all_model_scores = results_df.copy()
all_model_scores = pd.concat([all_model_scores, pd.DataFrame([tuned_metrics])], ignore_index=True)
all_model_scores = all_model_scores.sort_values('F1', ascending=False).reset_index(drop=True)

display(all_model_scores)

best_model_name = all_model_scores.loc[0, 'Model']
best_model = tuned_rf_model if best_model_name == 'RandomForest_Tuned' else fitted_models[best_model_name]

print('Final selected model:', best_model_name)

# Final predictions on test set
if hasattr(best_model, 'predict_proba'):
    best_prob = best_model.predict_proba(X_test)[:, 1]
else:
    raw_score = best_model.decision_function(X_test)
    best_prob = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min() + 1e-9)

best_pred = best_model.predict(X_test)

predictions_df = X_test.copy()
predictions_df['Actual_PitNextLap'] = y_test.values
predictions_df['Predicted_PitNextLap'] = best_pred
predictions_df['Predicted_Probability'] = best_prob

# Highest pit-risk laps according to model
high_risk = predictions_df.sort_values('Predicted_Probability', ascending=False).head(20)

display(high_risk)

print('Key insight: Rows with high tire-life/degradation and late-race progression often show elevated pit probability.')

In [ ]:
# Optional: simple feature importance view for interpretability (if tree-based model selected)
if 'RandomForest' in best_model_name:
    pre = best_model.named_steps['preprocessor']
    rf = best_model.named_steps['model']

    feature_names = pre.get_feature_names_out()
    importances = rf.feature_importances_

    fi_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).head(20)

    plt.figure(figsize=(10, 7))
    sns.barplot(data=fi_df, y='feature', x='importance', palette='viridis')
    plt.title('Top 20 Feature Importances (Final Model)')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.show()

    display(fi_df)
else:
    print('Final model is not tree-based; skipping tree feature importance plot.')

## Notebook End

This notebook is Kaggle-ready and can be extended with:
- Time-aware split by race order
- Threshold optimization for pit-call precision/recall trade-off
- Additional models (e.g., XGBoost/LightGBM)
- Race-specific strategy simulations